# StoryGAN Implementation

#### Importing Libraries and Preparing the Data

The github link provided to retrieve the CLEVR-SV dataset was problematic. Hence, I referred to sources for the correct way to import the data.

In [10]:
import numpy as np
import os
import glob
from collections import defaultdict
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from tqdm import tqdm
from transformers import CLIPTokenizer, CLIPTextModel

dataset_dir = "./clevr_dataset"
image_dir = os.path.join(dataset_dir, "images") 
npy_file = os.path.join(dataset_dir, "CLEVR_dict.npy")

if not os.path.exists(dataset_dir):
    raise FileNotFoundError(f"Dataset directory not found: {dataset_dir}")
if not os.path.exists(image_dir):
    raise FileNotFoundError(f"Image directory not found: {image_dir}")
if not os.path.exists(npy_file):
    raise FileNotFoundError(f".npy file not found: {npy_file}")

mappings = np.load(npy_file, allow_pickle=True, encoding='latin1').item()
if not isinstance(mappings, dict):
    raise ValueError("Expected a dictionary in CLEVR_dict.npy, got type: {}".format(type(mappings)))
print(f"Loaded {len(mappings)} mappings from CLEVR_dict.npy")

image_files = glob.glob(os.path.join(image_dir, "*.png")) 
stories = defaultdict(list)
for img_path in image_files:
    filename = os.path.basename(img_path) 
    story_id = filename.split("_")[2] 
    stories[story_id].append(filename)

story_data = []
skipped_missing_embeddings = []
skipped_wrong_image_count = []
for story_id in tqdm(stories, desc="Processing stories"):
    story_images = sorted(
        stories[story_id],
        key=lambda x: int(x.split("_")[3].split(".")[0])
    )
    if len(story_images) == 4:
        image_paths = [os.path.join(image_dir, img) for img in story_images]
        embeddings = [mappings.get(img, None) for img in story_images]
        if any(emb is None for emb in embeddings):
            missing_images = [img for img, emb in zip(story_images, embeddings) if emb is None]
            print(f"Warning: Missing embeddings for story {story_id}: {missing_images}")
            skipped_missing_embeddings.append(story_id)
            continue
        embeddings = [np.array(emb) for emb in embeddings]
        story_data.append({
            "story_id": story_id,
            "images": image_paths,  
            "embeddings": embeddings  
        })
    else:
        print(f"Warning: Story {story_id} has {len(story_images)} images, expected 4")
        skipped_wrong_image_count.append(story_id)


print(f"Total stories processed: {len(stories)}")
print(f"Valid stories included: {len(story_data)}")
print(f"Stories skipped (missing embeddings): {len(skipped_missing_embeddings)}")
print(f"Stories skipped (incorrect image count): {len(skipped_wrong_image_count)}")

transform = transforms.Compose([
    transforms.Resize((64, 64)),  
    transforms.ToTensor(),  
    transforms.Normalize([0.5], [0.5])  
])

def load_images(image_paths):
    images = []
    for img_path in image_paths:
        img = Image.open(img_path).convert("RGB") 
        img = transform(img) 
        images.append(img)
    return images

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")

def get_text_embeddings(sentences):
    inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        outputs = text_model(**inputs)
    return outputs.last_hidden_state  

class CLEVR_SVDataset(Dataset):
    def __init__(self, story_data, transform=None):
        self.story_data = story_data 
        self.transform = transform  

    def __len__(self):
        return len(self.story_data)  

    def __getitem__(self, idx):
        story = self.story_data[idx]
        image_paths = story["images"]  
        embeddings = story["embeddings"] 

        images = load_images(image_paths) 
        images = torch.stack(images) 

        embeddings = torch.from_numpy(np.stack(embeddings)).float() 

        return {
            "images": images,
            "text_embeddings": embeddings,
            "story_id": story["story_id"]
        }

dataset = CLEVR_SVDataset(story_data, transform=transform)
batch_size = 32  
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4)

if len(dataset) > 0:
    sample = dataset[0]
    print("\nTest Sample:")
    print("Story ID:", sample["story_id"])
    print("Image shapes:", sample["images"].shape)  
    print("Embedding shapes:", sample["text_embeddings"].shape) 
else:
    print("Error: No valid stories found in the dataset")


Loaded 384 mappings from CLEVR_dict.npy


Processing stories: 100%|██████████| 97/97 [00:00<00:00, 12040.11it/s]

Total stories processed: 97
Valid stories included: 96
Stories skipped (missing embeddings): 1
Stories skipped (incorrect image count): 0



Test Sample:
Story ID: 000148
Image shapes: torch.Size([4, 3, 64, 64])
Embedding shapes: torch.Size([4, 75])


#### Defining the Text2Gist Cell

I used the implementation provided in the implementation plan.

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Text2GistCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(Text2GistCell, self).__init__()
        self.z = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.r = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.h = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.filter = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Unflatten(1, (hidden_dim, 1))
        )

    def forward(self, it, h_prev):
        combined = torch.cat((it, h_prev), dim=-1)
        z_t = torch.sigmoid(self.z(combined))
        r_t = torch.sigmoid(self.r(combined))
        h_tilde = torch.tanh(self.h(torch.cat((it, r_t * h_prev), dim=-1)))
        h_t = (1 - z_t) * h_prev + z_t * h_tilde
        
        filt = self.filter(it)          
        h_conv = h_t.unsqueeze(-1)       
        o_t = torch.sum(filt * h_conv, dim=2)  
        return o_t, h_t

#### Defining the Story Encoder and the Context Encoder

I used a custom nerual network model for the Story Encoder and the Context Encoder.

In [12]:
class StoryEncoder(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        super(StoryEncoder, self).__init__()
        self.mlp_mu = nn.Sequential(
            nn.Linear(embedding_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.mlp_logvar = nn.Sequential(
            nn.Linear(embedding_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, sentence_embeddings):
        batch_size, seq_len, embed_dim = sentence_embeddings.size()
        S_enc = sentence_embeddings.view(batch_size, seq_len * embed_dim)
        mu = self.mlp_mu(S_enc)
        logvar = self.mlp_logvar(S_enc)
        h0 = self.reparameterize(mu, logvar)
        return h0, mu, logvar

class ContextEncoder(nn.Module):
    def __init__(self, sentence_dim, noise_dim, hidden_dim):
        super(ContextEncoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.gru = nn.GRUCell(sentence_dim + noise_dim, hidden_dim)
        self.text2gist = Text2GistCell(hidden_dim, hidden_dim)

    def forward(self, sentence_embeddings, noise):
        batch_size, seq_len = sentence_embeddings.size(0), sentence_embeddings.size(1)
        g_t = torch.zeros(batch_size, self.hidden_dim).to(sentence_embeddings.device)  
        h_t = torch.zeros(batch_size, self.hidden_dim).to(sentence_embeddings.device)  
        o_ts = []
        for t in range(seq_len):
            x_t = torch.cat((sentence_embeddings[:, t], noise[:, t]), dim=-1)
            g_t = self.gru(x_t, g_t) 
            o_t, h_t = self.text2gist(g_t, h_t) 
            o_ts.append(o_t)
        return torch.stack(o_ts, dim=1)

#### Defining the Image Generator and the Dual Discriminators

In [13]:
class ImageGenerator(nn.Module):
    def __init__(self, gist_dim, image_channels=3, image_size=64):
        super(ImageGenerator, self).__init__()
        self.gist_dim = gist_dim
        self.model = nn.Sequential(
            nn.Linear(gist_dim, 256 * 8 * 8),
            nn.ReLU(),
            nn.Unflatten(1, (256, 8, 8)),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), 
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), 
            nn.ReLU(),
            nn.ConvTranspose2d(64, image_channels, 4, stride=2, padding=1),  
            nn.Tanh()
        )

    def forward(self, o_t):
        batch_size, seq_len, gist_dim = o_t.size()
        o_t_flat = o_t.view(batch_size * seq_len, gist_dim)
        img_flat = self.model(o_t_flat)  
        images = img_flat.view(batch_size, seq_len, *img_flat.shape[1:])  
        return images

class ImageDiscriminator(nn.Module):
    def __init__(self, sentence_dim, h0_dim, image_channels=3, image_size=64):
        super(ImageDiscriminator, self).__init__()
        self.image_encoder = nn.Sequential(
            nn.Conv2d(image_channels, 64, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Flatten()
        )
        self.fc = nn.Linear(256 * 8 * 8 + sentence_dim + h0_dim, 1)

    def forward(self, x_t, s_t, h0):
        img_features = self.image_encoder(x_t)
        combined = torch.cat((img_features, s_t, h0), dim=-1)
        return torch.sigmoid(self.fc(combined))

class StoryDiscriminator(nn.Module):
    def __init__(self, sentence_dim, image_channels=3, image_size=64):
        super(StoryDiscriminator, self).__init__()
        self.image_encoder = nn.Sequential(
            nn.Conv2d(image_channels, 64, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Flatten()
        )
        self.text_encoder = nn.Linear(sentence_dim * 4, 512)
        self.fc = nn.Linear(256 * 8 * 8 * 4 + 512, 1)

    def forward(self, X, S):
        batch_size = X.size(0)
        img_features = []
        for t in range(4):
            img_feat = self.image_encoder(X[:, t])
            img_features.append(img_feat)
        img_features = torch.cat(img_features, dim=1)
        text_features = self.text_encoder(S.view(batch_size, -1))
        final_input = torch.cat((img_features, text_features), dim=1) 
        return torch.sigmoid(self.fc(final_input))



#### Tweaking the hyperparameters

The number of epochs should be preferable large, otherwise it may seem that the loss is significant. Tweaking the hyperparameters may improve the accuracy of the model.

In [14]:
embedding_dim = 75
hidden_dim = 512
noise_dim = 100
image_channels = 3
image_size = 64
num_epochs = 100
alpha = 1.0
beta = 1.0
lr_g = 0.0002
lr_d = 0.0002
beta1 = 0.5

#### Loss Functions and Optimizers and the initialization of the Models

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
story_encoder = StoryEncoder(embedding_dim, hidden_dim).to(device)
context_encoder = ContextEncoder(embedding_dim, noise_dim, hidden_dim).to(device)
generator = ImageGenerator(hidden_dim, image_channels, image_size).to(device)
image_discriminator = ImageDiscriminator(embedding_dim, hidden_dim, image_channels, image_size).to(device)
story_discriminator = StoryDiscriminator(embedding_dim, image_channels, image_size).to(device)

optimizer_g = torch.optim.Adam(
    list(story_encoder.parameters()) + list(context_encoder.parameters()) + list(generator.parameters()),
    lr=lr_g, betas=(beta1, 0.999)
)
optimizer_d = torch.optim.Adam(
    list(image_discriminator.parameters()) + list(story_discriminator.parameters()),
    lr=lr_d, betas=(beta1, 0.999)
)

bce_loss = nn.BCELoss()
def kl_divergence(mu, logvar):
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

#### Training the Model

In [16]:
print("Traiing Start")
for epoch in range(num_epochs):
    epoch_d_loss = 0
    epoch_g_loss = 0
    num_batches = 0
    
    for batch in dataloader:
        images = batch["images"].to(device) 
        text_embeddings = batch["text_embeddings"].to(device)  
        batch_size = images.size(0)
        real_label = torch.ones(batch_size, 1).to(device)
        fake_label = torch.zeros(batch_size, 1).to(device)
        optimizer_d.zero_grad()
        h0, mu, logvar = story_encoder(text_embeddings)
        
        image_d_real_loss = 0
        for t in range(4):
            d_out = image_discriminator(images[:, t], text_embeddings[:, t], h0)
            image_d_real_loss += bce_loss(d_out, real_label)
        image_d_real_loss /= 4

        noise = torch.randn(batch_size, 4, noise_dim).to(device)
        o_t = context_encoder(text_embeddings, noise)
        fake_images = generator(o_t)

        image_d_fake_loss = 0
        for t in range(4):
            d_out = image_discriminator(fake_images[:, t].detach(), text_embeddings[:, t], h0.detach())
            image_d_fake_loss += bce_loss(d_out, fake_label)
        image_d_fake_loss /= 4

        story_d_real = story_discriminator(images, text_embeddings)
        story_d_real_loss = bce_loss(story_d_real, real_label)
        story_d_fake = story_discriminator(fake_images.detach(), text_embeddings)
        story_d_fake_loss = bce_loss(story_d_fake, fake_label)
        d_loss = image_d_real_loss + image_d_fake_loss + story_d_real_loss + story_d_fake_loss
        d_loss.backward()
        optimizer_d.step()
        optimizer_g.zero_grad()
        
        noise = torch.randn(batch_size, 4, noise_dim).to(device)
        h0, mu, logvar = story_encoder(text_embeddings)
        o_t = context_encoder(text_embeddings, noise)
        fake_images = generator(o_t)
        image_g_loss = 0
        for t in range(4):
            d_out = image_discriminator(fake_images[:, t], text_embeddings[:, t], h0)
            image_g_loss += bce_loss(d_out, real_label)
        image_g_loss /= 4
        story_g = story_discriminator(fake_images, text_embeddings)
        story_g_loss = bce_loss(story_g, real_label)
        kl_loss = kl_divergence(mu, logvar)
        g_loss = alpha * image_g_loss + beta * story_g_loss + kl_loss
        g_loss.backward()
        optimizer_g.step()
        epoch_d_loss += d_loss.item()
        epoch_g_loss += g_loss.item()
        num_batches += 1

    avg_d_loss = epoch_d_loss / num_batches
    avg_g_loss = epoch_g_loss / num_batches
    print(f"Epoch [{epoch+1}/{num_epochs}], Avg D Loss: {avg_d_loss:.4f}, Avg G Loss: {avg_g_loss:.4f}")

print("Training End")

Traiing Start


Epoch [1/100], Avg D Loss: 2.6775, Avg G Loss: 1.5112
Epoch [2/100], Avg D Loss: 2.0658, Avg G Loss: 2.1457
Epoch [3/100], Avg D Loss: 0.8848, Avg G Loss: 4.9925
Epoch [4/100], Avg D Loss: 0.2754, Avg G Loss: 8.5050
Epoch [5/100], Avg D Loss: 4.4806, Avg G Loss: 6.4623
Epoch [6/100], Avg D Loss: 2.0344, Avg G Loss: 9.5018
Epoch [7/100], Avg D Loss: 0.6860, Avg G Loss: 6.5189
Epoch [8/100], Avg D Loss: 0.7936, Avg G Loss: 7.1488
Epoch [9/100], Avg D Loss: 0.4804, Avg G Loss: 8.0592
Epoch [10/100], Avg D Loss: 0.2646, Avg G Loss: 7.8872
Epoch [11/100], Avg D Loss: 0.3513, Avg G Loss: 8.7175
Epoch [12/100], Avg D Loss: 1.8435, Avg G Loss: 12.4741
Epoch [13/100], Avg D Loss: 5.7159, Avg G Loss: 7.7767
Epoch [14/100], Avg D Loss: 3.3724, Avg G Loss: 6.4057
Epoch [15/100], Avg D Loss: 2.6391, Avg G Loss: 6.1205
Epoch [16/100], Avg D Loss: 2.3021, Avg G Loss: 4.8484
Epoch [17/100], Avg D Loss: 1.3785, Avg G Loss: 5.2794
Epoch [18/100], Avg D Loss: 1.3508, Avg G Loss: 5.3382
Epoch [19/100], Av